In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
# Path() is better than hardcoded strings — works on Windows and Mac
PROJECT_ROOT = Path("..").resolve()
DATA_PATH    = PROJECT_ROOT / "data" / "metrics_159d.csv"
OUTPUT_PATH  = PROJECT_ROOT / "outputs" / "eda"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"]     = 120

print(f"Data file exists: {DATA_PATH.exists()}")
print(f"Output dir: {OUTPUT_PATH}")

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, parse_dates=["date"])

# Basic shape
print(f"Shape: {df.shape}")
print(f"\nDate range: {df['date'].min().date()}  →  {df['date'].max().date()}")
print(f"Total days: {(df['date'].max() - df['date'].min()).days + 1}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nNull counts:\n{df.isnull().sum()}")

In [ ]:
# ── Unique values per categorical column ──────────────────────────────────────
for col in ["source", "metric", "channel", "customer_type"]:
    vals = df[col].fillna("(empty)").unique()
    print(f"\n{col} ({len(vals)} unique):")
    for v in sorted(vals):
        n = df[df[col].fillna("(empty)") == v].shape[0]
        print(f"  {v:30s}  {n:6,} rows")

In [ ]:
# ── Row counts and value stats per source+metric ──────────────────────────────
summary = (
    df.groupby(["source", "metric"])["value"]
    .agg(count="count", min="min", max="max", mean="mean", std="std", median="median")
    .round(2)
    .reset_index()
)
pd.set_option("display.max_rows", 60)
display(summary)

In [ ]:
# ── Shopify daily totals: revenue ─────────────────────────────────────────────
# Long format → we must filter explicitly. No channel, no campaign, no customer_type
# means it's the "total" row per the data dictionary.

daily = (
    df[
        (df["source"] == "shopify") &
        (df["metric"] == "revenue") &
        (df["channel"].isna() | (df["channel"] == "")) &
        (df["customer_type"].isna() | (df["customer_type"] == ""))
    ]
    .groupby("date")["value"]
    .sum()
    .reset_index()
    .rename(columns={"value": "revenue"})
    .sort_values("date")
)

print(f"Daily revenue rows: {len(daily)}")
print(f"Missing dates: {159 - len(daily)}")
print(f"\nRevenue stats (INR):")
print(daily["revenue"].describe().apply(lambda x: f"₹{x:,.0f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(daily["date"], daily["revenue"], color="#1D9E75", linewidth=1.5, alpha=0.9)
ax.fill_between(daily["date"], daily["revenue"], alpha=0.08, color="#1D9E75")

# Format axes
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))  # Monday ticks
plt.xticks(rotation=30, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"₹{x/1e5:.1f}L"))

ax.set_title("Daily revenue — 159 days", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("Revenue (INR Lakhs)")

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "01_revenue_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")

In [ ]:
# ── Day-of-week patterns ───────────────────────────────────────────────────────
daily["dow"] = daily["date"].dt.day_name()
daily["dow_n"] = daily["date"].dt.dayofweek  # 0=Monday

dow_avg = (
    daily.groupby(["dow_n", "dow"])["revenue"]
    .agg(mean="mean", median="median", std="std")
    .reset_index()
    .sort_values("dow_n")
)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(dow_avg["dow"], dow_avg["mean"] / 1e5,
              color=["#1D9E75" if v == dow_avg["mean"].max() else "#9FE1CB"
                     for v in dow_avg["mean"]])
ax.errorbar(dow_avg["dow"], dow_avg["mean"] / 1e5,
            yerr=dow_avg["std"] / 1e5, fmt="none", color="gray", capsize=4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"₹{x:.1f}L"))
ax.set_title("Average daily revenue by day of week", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "02_dow_seasonality.png", dpi=150, bbox_inches="tight")
plt.show()

print(dow_avg[["dow", "mean", "std"]].to_string(index=False))

In [ ]:
# ── 28-day rolling mean and ±2σ bands ─────────────────────────────────────────
# This is the core statistical primitive your ranker will use in Phase 2

daily = daily.sort_values("date").reset_index(drop=True)
daily["roll_mean"] = daily["revenue"].rolling(28, min_periods=14).mean()
daily["roll_std"]  = daily["revenue"].rolling(28, min_periods=14).std()
daily["upper"]     = daily["roll_mean"] + 2 * daily["roll_std"]
daily["lower"]     = daily["roll_mean"] - 2 * daily["roll_std"]
daily["z_score"]   = (daily["revenue"] - daily["roll_mean"]) / daily["roll_std"]

# Flag anomalies (|z| > 2)
anomalies = daily[daily["z_score"].abs() > 2]
print(f"Anomaly days (|z| > 2): {len(anomalies)}")
print(anomalies[["date", "revenue", "roll_mean", "z_score"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily["date"], daily["revenue"] / 1e5, color="#1D9E75", lw=1.5, label="Revenue")
ax.plot(daily["date"], daily["roll_mean"] / 1e5, color="#0F6E56", lw=1, ls="--", label="28d mean")
ax.fill_between(daily["date"], daily["lower"] / 1e5, daily["upper"] / 1e5,
                alpha=0.12, color="#1D9E75", label="±2σ band")
ax.scatter(anomalies["date"], anomalies["revenue"] / 1e5,
           color="coral", s=60, zorder=5, label="Anomaly (|z|>2)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"₹{x:.1f}L"))
ax.legend(); ax.set_title("Revenue with rolling baseline and anomaly flags", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "03_anomaly_bands.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── WoW delta ─────────────────────────────────────────────────────────────────
daily["prev_week_revenue"] = daily["revenue"].shift(7)
daily["wow_delta_pct"] = (daily["revenue"] - daily["prev_week_revenue"]) / daily["prev_week_revenue"] * 100

# Last 4 weeks
last_28 = daily.tail(28)[["date", "dow", "revenue", "prev_week_revenue", "wow_delta_pct"]].copy()
last_28["revenue_L"] = last_28["revenue"] / 1e5
last_28["prev_L"]    = last_28["prev_week_revenue"] / 1e5

print("Last 28 days — WoW delta:")
print(last_28[["date", "dow", "revenue_L", "prev_L", "wow_delta_pct"]].round(1).to_string(index=False))

In [ ]:
# ── Correlate ad spend with revenue ───────────────────────────────────────────
meta_spend = (
    df[(df["source"] == "meta_ads") & (df["metric"] == "spend")]
    .groupby("date")["value"].sum().rename("meta_spend")
)
google_spend = (
    df[(df["source"] == "google_ads") & (df["metric"] == "spend")]
    .groupby("date")["value"].sum().rename("google_spend")
)

combined = (
    daily.set_index("date")[["revenue"]]
    .join(meta_spend)
    .join(google_spend)
    .dropna()
)

print("Correlation matrix:")
print(combined.corr().round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes, ["meta_spend", "google_spend"], ["Meta", "Google"]):
    ax.scatter(combined[col] / 1e5, combined["revenue"] / 1e5, alpha=0.4, s=20)
    ax.set_xlabel(f"{label} spend (₹L)"); ax.set_ylabel("Revenue (₹L)")
    ax.set_title(f"{label} spend vs revenue")
    m, b = np.polyfit(combined[col], combined["revenue"], 1)
    x_line = np.linspace(combined[col].min(), combined[col].max(), 100)
    ax.plot(x_line / 1e5, (m * x_line + b) / 1e5, "r--", lw=1)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "04_spend_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Structured summary to paste into design doc ───────────────────────────────
summary_text = f"""
## EDA Key Findings — metrics_159d.csv

**Dataset overview**
- Date range: {daily['date'].min().date()} to {daily['date'].max().date()} ({len(daily)} days)
- Sources: meta_ads, google_ads, shopify
- Total rows: {len(df):,}

**Revenue (INR)**
- Median daily: ₹{daily['revenue'].median():,.0f}
- Min: ₹{daily['revenue'].min():,.0f}  Max: ₹{daily['revenue'].max():,.0f}
- Best day of week (avg): {dow_avg.loc[dow_avg['mean'].idxmax(), 'dow']}
- Worst day of week (avg): {dow_avg.loc[dow_avg['mean'].idxmin(), 'dow']}

**Anomaly detection (28d rolling)**
- Anomaly days (|z|>2): {len(anomalies)}
- Dates: {', '.join(str(d.date()) for d in anomalies['date'])}

**Ad spend correlation with revenue**
- Meta r: {combined[['meta_spend','revenue']].corr().iloc[0,1]:.3f}
- Google r: {combined[['google_spend','revenue']].corr().iloc[0,1]:.3f}

**Attribution caveat**
- ~91% of shopify orders are 'unattributed' at referrer-channel level
- Channel slice is directional only; do not use for causal statements
"""

print(summary_text)

with open(OUTPUT_PATH / "eda_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)
print("Saved to outputs/eda/eda_summary.txt")

In [ ]:
# ── Verify the shopify total filter is correct ────────────────────────────────
# This catches the triple-counting bug described earlier

# How many rows match our "daily total" filter?
total_rows = df[
    (df["source"] == "shopify") &
    (df["metric"] == "revenue") &
    (df["channel"].isna() | (df["channel"] == "")) &
    (df["customer_type"].isna() | (df["customer_type"] == ""))
]

# How many rows exist for shopify revenue total?
all_shopify_rev = df[
    (df["source"] == "shopify") &
    (df["metric"] == "revenue")
]

print(f"All shopify revenue rows : {len(all_shopify_rev)}")
print(f"After total-only filter  : {len(total_rows)}")
print(f"Unique dates in total    : {total_rows['date'].nunique()}")
print(f"Expected (160 days)      : 160")
print()

# Show the breakdown of channel values for shopify revenue
print("Channel breakdown for shopify revenue rows:")
print(
    df[(df["source"]=="shopify") & (df["metric"]=="revenue")]["channel"]
    .fillna("(null)").value_counts().to_string()
)